In [ ]:
import pandas as pd
import os

df = pd.read_csv("fes_raw/ES1 Supply 2025.csv")

keep_scenarios = ["Electric Engagement", "Holistic Transition"]

df = df[df["Pathway"].isin(keep_scenarios)].copy()

df = df[df["Variable"].eq("Generation (TWh)")].copy()

type_to_tech = {
    "Biomass": "biomass",
    "Waste": "waste",
    "Gas": "gas",
    "Other Thermal": "other_thermal",
    "Hydro": "hydro",
    "Offshore Wind": "wind_total",
    "Onshore Wind": "wind_total",
    "Other Renewable": "other_renewable",
    "Solar PV": "solar",
    "Battery": "storage",
    "Long Duration Energy Storage": "storage",
    "Hydrogen": "hydrogen",
    "CCS Biomass": "biomass_ccs",
    "Nuclear": "nuclear",
    "CCS Gas": "gas_ccs",
}

df["tech"] = df["Type"].map(type_to_tech)

unmapped = sorted(df.loc[df["tech"].isna(), "Type"].dropna().unique())
assert not unmapped, f"Unmapped FES supply Type values: {unmapped}"

years = [str(y) for y in range(2030, 2046)]
years_exist = [y for y in years if y in df.columns]

df_long = df.melt(
    id_vars=["Pathway", "Category", "Type", "SubType", "tech", "Variable"],
    value_vars=years_exist,
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].astype(int)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.dropna(subset=["value"]).copy()

df_long = df_long.rename(columns={"Pathway": "fes_scenario"})

df_long["unit"] = "TWh"
df_long["metric_type"] = "generation"

final_output = (
    df_long
    .groupby(["year", "fes_scenario", "tech", "metric_type"], as_index=False)
    .agg(
        value=("value", "sum"),
        unit=("unit", "first")
    )
)

final_output = final_output[
    ["year", "fes_scenario", "tech", "value", "unit", "metric_type"]
].sort_values(["fes_scenario", "year", "tech"]).reset_index(drop=True)

assert final_output["year"].min() == 2030
assert final_output["year"].max() == 2045
assert set(final_output["fes_scenario"]) == set(keep_scenarios)
assert not final_output.duplicated(["year", "fes_scenario", "tech", "metric_type"]).any()

os.makedirs("output", exist_ok=True)

final_output.to_parquet(
    "output/fes_supply_annual_2030_2045.parquet",
    index=False
)

print("FES Supply processing complete")
print("Rows:", len(final_output))
print("Duplicate keys:", final_output.duplicated(["year", "fes_scenario", "tech", "metric_type"]).sum())
print("Columns:", final_output.columns.tolist())
print("Scenarios:", sorted(final_output["fes_scenario"].unique()))
print("Years:", final_output["year"].min(), "~", final_output["year"].max())
print("Tech:", sorted(final_output["tech"].unique()))

In [ ]:
import pandas as pd
import os

df_demand = pd.read_csv("fes_raw/ES1 Demand 2025.csv")

scenarios_we_need = ["Electric Engagement", "Holistic Transition"]
years = [str(y) for y in range(2030, 2046)]

annual = df_demand[
    (df_demand["Pathway"].isin(scenarios_we_need)) &
    (df_demand["Data item"].astype(str).str.strip().eq("GBFES System Demand: Total")) &
    (df_demand["Peak/ Annual/ Minimum"].astype(str).str.strip().eq("Annual [Fiscal]")) &
    (df_demand["Unit"].astype(str).str.strip().eq("GWh"))
].copy()

annual_long = annual.melt(
    id_vars=["Pathway"],
    value_vars=years,
    var_name="year",
    value_name="demand_gwh"
)

annual_long["year"] = annual_long["year"].astype(int)
annual_long["demand_gwh"] = pd.to_numeric(annual_long["demand_gwh"], errors="coerce")
annual_long["annual_demand_twh"] = annual_long["demand_gwh"] / 1000
annual_long = annual_long.rename(columns={"Pathway": "fes_scenario"})

peak = df_demand[
    (df_demand["Pathway"].isin(scenarios_we_need)) &
    (df_demand["Data item"].astype(str).str.strip().eq("GBFES Peak Customer Demand: Total Consumption plus Losses")) &
    (df_demand["Peak/ Annual/ Minimum"].astype(str).str.strip().eq("Peak")) &
    (df_demand["Unit"].astype(str).str.strip().eq("GW"))
].copy()

peak_long = peak.melt(
    id_vars=["Pathway"],
    value_vars=years,
    var_name="year",
    value_name="peak_demand_gw"
)

peak_long["year"] = peak_long["year"].astype(int)
peak_long["peak_demand_gw"] = pd.to_numeric(peak_long["peak_demand_gw"], errors="coerce")
peak_long = peak_long.rename(columns={"Pathway": "fes_scenario"})

final_output = annual_long.merge(
    peak_long[["year", "fes_scenario", "peak_demand_gw"]],
    on=["year", "fes_scenario"],
    how="left",
    validate="one_to_one"
)

final_output = final_output[
    ["year", "fes_scenario", "annual_demand_twh", "peak_demand_gw"]
].sort_values(["fes_scenario", "year"]).reset_index(drop=True)

assert final_output["year"].min() == 2030
assert final_output["year"].max() == 2045
assert set(final_output["fes_scenario"]) == set(scenarios_we_need)
assert final_output["annual_demand_twh"].notna().all()
assert final_output["annual_demand_twh"].gt(0).all()
assert final_output["peak_demand_gw"].notna().all()
assert final_output["peak_demand_gw"].gt(0).all()
assert not final_output.duplicated(["year", "fes_scenario"]).any()

os.makedirs("output", exist_ok=True)

final_output.to_parquet(
    "output/fes_demand_annual_2030_2045.parquet",
    index=False
)

print("FES Demand generated")
print("Rows:", len(final_output))
print("Duplicate keys:", final_output.duplicated(["year", "fes_scenario"]).sum())
print("Columns:", final_output.columns.tolist())
print("Scenarios:", sorted(final_output["fes_scenario"].unique()))
print("Years:", final_output["year"].min(), "~", final_output["year"].max())
print("Peak demand min/max:", final_output["peak_demand_gw"].min(), final_output["peak_demand_gw"].max())
display(final_output.head())